# Módulo 8 · Clase 5 — Redes Densas: Clasificación

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Recapitulación: Dense layers en profundidad |
| 2 | Clasificación binaria: arquitectura y entrenamiento |
| 3 | Clasificación multiclase: softmax y one-hot |
| 4 | ¿Qué aprende la red? Visualizaciones de pesos |
| 5 | Diagnóstico y ajuste de hiperparámetros |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification, make_blobs

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. Dense layers: qué ocurre dentro

```
layers.Dense(n_neuronas, activation='relu')

  Internamente almacena:
    W: matriz (n_entrada × n_neuronas)    ← pesos, inicializados con Glorot/He
    b: vector (n_neuronas,)               ← bias, inicializado en cero

  Forward pass para un batch X de shape (batch, n_entrada):
    z = X @ W + b      → shape: (batch, n_neuronas)
    a = f(z)           → idem, aplicando la activación elemento a elemento

  Parámetros totales de una capa Dense(n_s):
    n_entrada × n_s  +  n_s  =  n_s × (n_entrada + 1)

  Ejemplo: Dense(64) después de Input(shape=(784,)):
    784 × 64 + 64 = 50,240 parámetros
```

In [ ]:
# Acceder a los pesos de una capa
modelo_demo = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(3, activation='relu', name='oculta'),
    layers.Dense(2, activation='softmax', name='salida'),
])

# Antes de entrenar los pesos ya existen (Glorot uniform por defecto)
for capa in modelo_demo.layers:
    pesos = capa.get_weights()
    if pesos:
        W, b = pesos
        print(f"Capa '{capa.name}':")
        print(f"  W: {W.shape}  b: {b.shape}")
        print(f"  W media={W.mean():.4f}, std={W.std():.4f}")
        print(f"  W[:2,:] = {W[:2]}\n")

# Forward pass manual vs Keras
x_sample = np.array([[1.0, 0.5, -0.3, 0.8]])
W1, b1 = modelo_demo.get_layer('oculta').get_weights()
W2, b2 = modelo_demo.get_layer('salida').get_weights()

z1 = x_sample @ W1 + b1
a1 = np.maximum(0, z1)  # ReLU manual
z2 = a1 @ W2 + b2
e  = np.exp(z2 - z2.max())
a2_manual = e / e.sum()

a2_keras = modelo_demo.predict(x_sample, verbose=0)
print("Forward pass:")
print(f"  Manual: {a2_manual.ravel().round(6)}")
print(f"  Keras:  {a2_keras.ravel().round(6)}")
print(f"  Diferencia max: {np.abs(a2_manual - a2_keras).max():.2e}")

---
## 2. Clasificación binaria

In [ ]:
# Dataset: crédito bancario simulado
# Objetivo: predecir si un cliente pagará su préstamo (1) o no (0)
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=6,
    n_redundant=2, weights=[0.7, 0.3], random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

print(f"Train: {X_tr_n.shape}, Test: {X_te_n.shape}")
print(f"Distribución train: {np.bincount(y_tr)} (clase 0 y 1)")

In [ ]:
# Regla para clasificación binaria:
# - Última capa: Dense(1, activation='sigmoid')
# - Loss: 'binary_crossentropy'
# - Salida: probabilidad [0, 1] → umbral 0.5 → clase 0 o 1

tf.random.set_seed(42)
modelo_bin = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
], name='clasificador_binario')

modelo_bin.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

cbs = [
    keras.callbacks.EarlyStopping('val_auc', patience=15, restore_best_weights=True, mode='max'),
]

hist_bin = modelo_bin.fit(
    X_tr_n, y_tr,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=cbs, verbose=0
)

loss_te, acc_te, auc_te = modelo_bin.evaluate(X_te_n, y_te, verbose=0)
print(f"Test  → Loss: {loss_te:.4f}, Accuracy: {acc_te:.4f}, AUC: {auc_te:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

# Predicciones
probas = modelo_bin.predict(X_te_n, verbose=0).ravel()
y_pred = (probas >= 0.5).astype(int)

print("Reporte de clasificación:")
print(classification_report(y_te, y_pred, target_names=['No paga', 'Paga']))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Confusion matrix
cm_val = confusion_matrix(y_te, y_pred)
axes[0].imshow(cm_val, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm_val[i,j], ha='center', va='center',
                     fontsize=20, color='white' if cm_val[i,j] > cm_val.max()/2 else 'black')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No paga','Paga']); axes[0].set_yticklabels(['No paga','Paga'])
axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de confusión')

# Distribución de probabilidades
axes[1].hist(probas[y_te==0], bins=25, alpha=0.6, label='No paga (real)', color='coral')
axes[1].hist(probas[y_te==1], bins=25, alpha=0.6, label='Paga (real)',    color='steelblue')
axes[1].axvline(0.5, color='black', ls='--', label='Umbral 0.5')
axes[1].set_xlabel('Probabilidad predicha'); axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de probabilidades')
axes[1].legend()

# Curva ROC
fpr, tpr, _ = roc_curve(y_te, probas)
axes[2].plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC = {auc_te:.4f}')
axes[2].plot([0,1],[0,1], 'k--', lw=1, label='Aleatorio (AUC=0.5)')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title('Curva ROC')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 3. Clasificación multiclase

```
Dos tipos de etiquetas:

  Enteras (sparse):   y = [0, 2, 1, 3, ...]      → loss='sparse_categorical_crossentropy'
  One-hot:            y = [[1,0,0], [0,0,1], ...]  → loss='categorical_crossentropy'

Última capa siempre:  Dense(n_clases, activation='softmax')

  softmax(zᵢ) = exp(zᵢ) / Σ exp(zⱼ)    → suma siempre = 1
```

In [ ]:
# MNIST: 10 clases de dígitos escritos a mano
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()

X_tr_m = X_tr_raw.reshape(-1, 784).astype('float32') / 255.0
X_te_m = X_te_raw.reshape(-1, 784).astype('float32') / 255.0

print(f"Train: {X_tr_m.shape}, Test: {X_te_m.shape}")
print(f"Clases: {np.unique(y_tr_raw)}")

# Ver algunas imágenes
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
idx = np.random.choice(len(X_tr_m), 16, replace=False)
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X_tr_m[i].reshape(28,28), cmap='gray')
    ax.set_title(str(y_tr_raw[i]), fontsize=12)
    ax.axis('off')
plt.suptitle('Muestras de MNIST', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Modelo para MNIST
tf.random.set_seed(42)
modelo_mnist = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),  # 10 clases
], name='mnist_dense')

modelo_mnist.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # y son enteros 0-9
    metrics=['accuracy']
)
modelo_mnist.summary()

hist_mnist = modelo_mnist.fit(
    X_tr_m, y_tr_raw,
    epochs=15, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_mnist = modelo_mnist.evaluate(X_te_m, y_te_raw, verbose=0)
print(f"\nTest accuracy MNIST: {acc_mnist:.4f} ({acc_mnist:.2%})")

In [ ]:
# Análisis de errores
y_pred_m = modelo_mnist.predict(X_te_m, verbose=0).argmax(axis=1)

# Confusion matrix
cm_m = confusion_matrix(y_te_raw, y_pred_m)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_m, cmap='Blues')
for i in range(10):
    for j in range(10):
        color = 'white' if cm_m[i,j] > cm_m.max()/2 else 'black'
        ax.text(j, i, cm_m[i,j], ha='center', va='center', fontsize=8, color=color)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusión MNIST (acc={acc_mnist:.4f})')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# Ver errores típicos
errores = np.where(y_pred_m != y_te_raw)[0]
print(f"Errores totales: {len(errores)} / {len(y_te_raw)} ({len(errores)/len(y_te_raw):.2%})")

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), errores[:18]):
    ax.imshow(X_te_m[idx].reshape(28,28), cmap='gray')
    ax.set_title(f'Real:{y_te_raw[idx]} Pred:{y_pred_m[idx]}',
                 color='red', fontsize=9)
    ax.axis('off')
plt.suptitle('Ejemplos de errores', fontsize=14)
plt.tight_layout(); plt.show()

---
## 4. ¿Qué aprende la red? Visualización de pesos

In [ ]:
# Los pesos de la primera capa (784 → 256) son filtros sobre el espacio de pixels
# Cada fila de W1 es un 'detector' que activa para ciertos patrones

W1 = modelo_mnist.layers[0].get_weights()[0]  # shape: (784, 256)
print(f"Pesos primera capa: {W1.shape}")

# Visualizar los primeros 32 'detectores'
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    # Cada columna de W1 es un detector de 784 dimensiones → 28x28
    detector = W1[:, i].reshape(28, 28)
    ax.imshow(detector, cmap='RdBu_r', vmin=detector.min(), vmax=detector.max())
    ax.axis('off')
plt.suptitle('Primeros 32 detectores de la capa 1 (azul=activa, rojo=inhibe)', fontsize=12)
plt.tight_layout(); plt.show()

# La última capa (64 → 10) son los 'prototipos' de cada dígito
# Podemos ver qué patrón de activación de la capa 64 corresponde a cada clase
W_out = modelo_mnist.layers[-1].get_weights()[0]  # (64, 10)
print(f"\nPesos capa de salida: {W_out.shape}")
print("(64 características → 10 clases)")

In [ ]:
# Qué activa cada neurona: modelo truncado para ver activaciones intermedias
modelo_intermedio = keras.Model(
    inputs=modelo_mnist.input,
    outputs=modelo_mnist.layers[0].output   # salida de la primera Dense
)

# Tomar un dígito de cada clase y ver sus activaciones
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for clase in range(10):
    idx = np.where(y_te_raw == clase)[0][0]
    img = X_te_m[idx:idx+1]
    act = modelo_intermedio.predict(img, verbose=0).ravel()

    ax = axes.ravel()[clase]
    ax.bar(range(len(act)), act, color='steelblue', alpha=0.7)
    ax.set_title(f'Dígito {clase}  (max={act.max():.2f})', fontsize=9)
    ax.set_xlabel('Neurona'); ax.set_ylabel('Activación')
    ax.tick_params(labelsize=7)

plt.suptitle('Activaciones de la capa 1 para cada dígito', fontsize=13)
plt.tight_layout(); plt.show()

---
## 5. Diagnóstico y ajuste de hiperparámetros

In [ ]:
# Los hiperparámetros más importantes en una red densa:
print("""
┌─────────────────────┬───────────────────┬─────────────────────────────┐
│ Hiperparámetro      │ Valores típicos   │ Efecto                      │
├─────────────────────┼───────────────────┼─────────────────────────────┤
│ n_capas             │ 1 – 5             │ ↑ = más capacidad           │
│ n_neuronas          │ 16 – 1024         │ ↑ = más capacidad           │
│ activación          │ relu, tanh, elu   │ relu por defecto            │
│ learning_rate       │ 1e-4 – 1e-2       │ ↑ = aprendizaje rápido/inestable │
│ batch_size          │ 16 – 512          │ ↑ = más estable, menos ruidoso   │
│ optimizer           │ adam, sgd+mom     │ adam por defecto            │
└─────────────────────┴───────────────────┴─────────────────────────────┘
""")

# Comparar learning rates
resultados_lr = {}
for lr in [0.01, 0.001, 0.0001]:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(10,  activation='softmax'),
    ])
    m.compile(keras.optimizers.Adam(lr), 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr_m, y_tr_raw, epochs=10, batch_size=128,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te_m, y_te_raw, verbose=0)
    resultados_lr[lr] = {'hist': h, 'acc': acc}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colores = ['steelblue', 'darkorange', 'green']
for (lr, res), color in zip(resultados_lr.items(), colores):
    axes[0].plot(res['hist'].history['val_loss'], color=color, lw=2, label=f'lr={lr}')
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2, label=f'lr={lr} (acc={res["acc"]:.3f})')
for ax in axes:
    ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Val Loss'); axes[0].set_title('Efecto del Learning Rate — Loss')
axes[1].set_ylabel('Val Accuracy'); axes[1].set_title('Efecto del Learning Rate — Accuracy')
plt.tight_layout(); plt.show()

In [ ]:
# Diagnóstico: underfitting vs overfitting vs buen ajuste
def diagnosticar(historial, acc_te):
    hist = historial.history
    acc_tr_final = hist['accuracy'][-1]
    acc_val_final = hist['val_accuracy'][-1]
    gap = acc_tr_final - acc_val_final

    print(f"  Train accuracy final: {acc_tr_final:.4f}")
    print(f"  Val accuracy final:   {acc_val_final:.4f}")
    print(f"  Test accuracy:        {acc_te:.4f}")
    print(f"  Gap train-val:        {gap:.4f}")

    if acc_tr_final < 0.80:
        print("  → UNDERFITTING: la red no tiene suficiente capacidad o entrena poco")
        print("     Solución: más neuronas, más capas, más épocas")
    elif gap > 0.05:
        print("  → OVERFITTING: la red memoriza el train, no generaliza")
        print("     Solución: Dropout, L2, Early Stopping, más datos")
    else:
        print("  → BUEN AJUSTE: train y val convergentes, buen rendimiento")

print("Diagnóstico del modelo MNIST:")
diagnosticar(hist_mnist, acc_mnist)

---
## Práctica

### Ejercicio 1 — Fashion MNIST

**Enunciado:** Fashion MNIST tiene 10 categorías de ropa (camiseta, pantalón, etc.).

```python
(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
clases = ['Camiseta','Pantalón','Jersey','Vestido','Abrigo',
          'Sandalia','Camisa','Zapatilla','Bolso','Botín']
```

1. Normaliza y aplana las imágenes
2. Construye una red densa de al menos 3 capas
3. Entrena con EarlyStopping (patience=8)
4. Muestra la matriz de confusión con los nombres de las clases
5. ¿Qué categorías se confunden más entre sí?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
clases = ['Camiseta','Pantalón','Jersey','Vestido','Abrigo',
          'Sandalia','Camisa','Zapatilla','Bolso','Botín']

X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

hist = m.fit(X_tr, y_tr, epochs=50, batch_size=128, validation_split=0.1,
             callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=8,
                                                       restore_best_weights=True)], verbose=0)
_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Test accuracy: {acc:.4f}")

y_pred = m.predict(X_te, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks(range(10)); ax.set_xticklabels(clases, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(clases)
ax.set_title(f'Fashion MNIST — acc={acc:.4f}')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# Clases con más confusión
np.fill_diagonal(cm, 0)
idx = np.unravel_index(cm.argmax(), cm.shape)
print(f"Confusión más frecuente: {clases[idx[0]]} confundido con {clases[idx[1]]} ({cm[idx]} veces)")
```
</details>

---
### Ejercicio 2 — Umbral de decisión

**Enunciado:** En el modelo de crédito bancario, el umbral por defecto es 0.5. Pero en contextos reales el coste de un falso negativo (prestar a quien no pagará) es mayor que el de un falso positivo.

1. Usa el modelo `modelo_bin` ya entrenado
2. Calcula precision, recall y F1 para umbrales: 0.2, 0.3, 0.4, 0.5, 0.6, 0.7
3. Plóta las métricas vs umbral
4. ¿Qué umbral maximiza el recall de la clase 0 ("no paga")?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

probas = modelo_bin.predict(X_te_n, verbose=0).ravel()
umbrales = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
metricas = []

for u in umbrales:
    y_pred_u = (probas >= u).astype(int)
    metricas.append({
        'umbral': u,
        'precision_c1': precision_score(y_te, y_pred_u, pos_label=1, zero_division=0),
        'recall_c1':    recall_score(y_te, y_pred_u, pos_label=1, zero_division=0),
        'precision_c0': precision_score(y_te, y_pred_u, pos_label=0, zero_division=0),
        'recall_c0':    recall_score(y_te, y_pred_u, pos_label=0, zero_division=0),
        'f1':           f1_score(y_te, y_pred_u),
    })

us = [m['umbral'] for m in metricas]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(us, [m['precision_c1'] for m in metricas], 'o-', label='Precision (clase 1)')
axes[0].plot(us, [m['recall_c1']    for m in metricas], 's-', label='Recall (clase 1)')
axes[0].plot(us, [m['f1']           for m in metricas], '^-', label='F1')
axes[0].set_xlabel('Umbral'); axes[0].set_title('Clase 1 (Paga)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(us, [m['precision_c0'] for m in metricas], 'o-', color='coral', label='Precision (clase 0)')
axes[1].plot(us, [m['recall_c0']    for m in metricas], 's-', color='red',   label='Recall (clase 0)')
axes[1].set_xlabel('Umbral'); axes[1].set_title('Clase 0 (No paga)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Umbral con mayor recall de clase 0
mejor = max(metricas, key=lambda m: m['recall_c0'])
print(f"Mayor recall clase 0: umbral={mejor['umbral']} → recall={mejor['recall_c0']:.4f}")
print("(menor umbral = más agresivo detectando 'no paga', más FP también)")
```
</details>

---
### Ejercicio 3 — Profundidad vs anchura

**Enunciado:** ¿Es mejor una red profunda (muchas capas, pocas neuronas) o ancha (pocas capas, muchas neuronas)?

Con MNIST, compara:
- Red profunda: 784 → 64 → 64 → 64 → 64 → 10 (~4 capas ocultas)
- Red ancha:    784 → 512 → 512 → 10 (~2 capas ocultas)
- Controla que tengan parámetros similares

¿Cuál converge más rápido? ¿Cuál generaliza mejor?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configs = [
    ('Profunda (4×64)',  [64, 64, 64, 64]),
    ('Ancha (2×512)',    [512, 512]),
]

resultados = {}
for nombre, capas_h in configs:
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in capas_h:
        capas.append(layers.Dense(n, activation='relu'))
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    print(f"{nombre}: {m.count_params():,} parámetros")
    h = m.fit(X_tr, y_tr, epochs=15, batch_size=128,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    resultados[nombre] = {'hist': h, 'acc': acc}
    print(f"  Test acc: {acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = ['steelblue', 'darkorange']
for (nombre, res), color in zip(resultados.items(), colores):
    axes[0].plot(res['hist'].history['val_loss'], color=color, lw=2, label=nombre)
    axes[1].plot(res['hist'].history['val_accuracy'], color=color, lw=2,
                 label=f'{nombre} ({res["acc"]:.3f})')
for ax in axes:
    ax.set_xlabel('Época'); ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Loss'); axes[1].set_title('Val Accuracy')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 4 — One-hot encoding

**Enunciado:** `sparse_categorical_crossentropy` acepta y como enteros. `categorical_crossentropy` requiere one-hot.

1. Convierte y_tr_raw y y_te_raw a one-hot con `keras.utils.to_categorical`
2. Construye el mismo modelo MNIST pero usando `categorical_crossentropy`
3. Verifica que obtiene el mismo accuracy que el modelo anterior

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr_int), (X_te, y_te_int) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

# One-hot encoding
y_tr_oh = keras.utils.to_categorical(y_tr_int, num_classes=10)
y_te_oh = keras.utils.to_categorical(y_te_int, num_classes=10)
print(f"y_tr shape con one-hot: {y_tr_oh.shape}")
print(f"Ejemplo: entero={y_tr_int[0]}, one-hot={y_tr_oh[0]}")

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
# categorical_crossentropy espera one-hot
m.compile('adam', 'categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr_oh, epochs=10, batch_size=128, validation_split=0.1, verbose=1)
_, acc = m.evaluate(X_te, y_te_oh, verbose=0)
print(f"\nTest accuracy: {acc:.4f}")
print("(debe ser muy similar al modelo con sparse_categorical_crossentropy)")
```
</details>

---
### Ejercicio 5 (reto) — Clasificador personalizable

**Enunciado:** Construye una función `crear_clasificador(n_entrada, capas_ocultas, n_clases, lr=0.001)` que:

1. Cree un modelo Sequential con la arquitectura especificada en `capas_ocultas` (lista de enteros)
2. Seleccione automáticamente la loss y la activación final según `n_clases` (1 → binario, >2 → multiclase)
3. Incluya métricas adecuadas para cada caso
4. Pruébala en:
   - make_classification(n_features=20) con capas_ocultas=[64, 32]
   - MNIST con capas_ocultas=[256, 128, 64]

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def crear_clasificador(n_entrada, capas_ocultas, n_clases, lr=0.001):
    capas = [layers.Input(shape=(n_entrada,))]
    for n in capas_ocultas:
        capas.append(layers.Dense(n, activation='relu'))

    if n_clases == 2:    # binario
        capas.append(layers.Dense(1, activation='sigmoid'))
        loss    = 'binary_crossentropy'
        metricas = ['accuracy', keras.metrics.AUC(name='auc')]
    else:                # multiclase
        capas.append(layers.Dense(n_clases, activation='softmax'))
        loss    = 'sparse_categorical_crossentropy'
        metricas = ['accuracy']

    m = keras.Sequential(capas)
    m.compile(keras.optimizers.Adam(lr), loss, metricas)
    return m

# Prueba 1: binario
X, y = make_classification(2000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, 0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr); X_te_n = sc.transform(X_te)

tf.random.set_seed(42)
m1 = crear_clasificador(20, [64, 32], n_clases=2)
m1.fit(X_tr_n, y_tr, epochs=20, batch_size=64, validation_split=0.1, verbose=0)
resultados = m1.evaluate(X_te_n, y_te, verbose=0)
print("Binario:", {m: f'{v:.4f}' for m, v in zip(['loss','acc','auc'], resultados)})

# Prueba 2: multiclase MNIST
(X_tr_m, y_tr_m), (X_te_m, y_te_m) = keras.datasets.mnist.load_data()
X_tr_m = X_tr_m.reshape(-1, 784).astype('float32') / 255.0
X_te_m = X_te_m.reshape(-1, 784).astype('float32') / 255.0

tf.random.set_seed(42)
m2 = crear_clasificador(784, [256, 128, 64], n_clases=10)
m2.fit(X_tr_m, y_tr_m, epochs=10, batch_size=128, validation_split=0.1, verbose=0)
_, acc = m2.evaluate(X_te_m, y_te_m, verbose=0)
print(f"Multiclase MNIST: acc={acc:.4f}")
```
</details>

---
## Resumen

```python
# Clasificación BINARIA
modelo = keras.Sequential([
    layers.Input(shape=(n,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1,  activation='sigmoid'),   # ← sigmoid
])
modelo.compile('adam', 'binary_crossentropy', ['accuracy', keras.metrics.AUC()])
probas  = modelo.predict(X_te)               # → valores [0, 1]
y_pred  = (probas >= 0.5).astype(int)        # umbral ajustable

# Clasificación MULTICLASE
modelo = keras.Sequential([
    layers.Input(shape=(n,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(n_clases, activation='softmax'),  # ← softmax
])
modelo.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])  # y enteros
# o 'categorical_crossentropy' si y es one-hot (keras.utils.to_categorical)
probas = modelo.predict(X_te)   # → (N, n_clases)
y_pred = probas.argmax(axis=1)  # → clase con mayor probabilidad

# Diagnóstico: train_acc >> val_acc → overfitting (próxima clase: Regularización)
# Acceder a pesos: modelo.layers[i].get_weights() → [W, b]
# Modelo intermedio: keras.Model(inputs=m.input, outputs=m.layers[i].output)
```

## ¿Qué viene en la próxima clase?
En la **Clase 6** aprenderemos a controlar el overfitting con **Regularización**: Dropout, Batch Normalization, L1/L2 y Early Stopping.